# CS 195: Natural Language Processing
## Alignment with Preference Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F7_2_Alignment.ipynb)


## End of year business

**Final Demos:** We will do one final day of demos on **Wednesday, May 6th** during our normal class time. There will be time to present more than one project if you need to do that.
* reminder: any creative synthesis project needs to be presented to at least a small group in order to claim credit

**Final Learning Reflection:** I will also set aside time on Wednesday, May 6th for a final learning reflection. It is ungraded but will help me see if the class structure promoted student learning, and how it can be improved. You can do it at any time if you like: https://docs.google.com/forms/d/e/1FAIpQLSeFd1tXM2W1dk57d6GooI1aavaQwfFDfGNfCqfQ7OgVjqGqkA/viewform

**Course Evaluations:** I will try to make time for this on either Monday or Wednesday of next week, but the link is on Blackboard if you want to get started early.

**Final Portfolios:** Final Portfolios are due by the end of the calendar day on Tuesday, May 12th. You must submit a new link to the assignment submission form on Blackboard. I will not assume that the links you submitted previously are the right ones.

## References

- [SLP: Post-training: Instruction Tuning, Alignment, and Test-Time Compute](https://web.stanford.edu/~jurafsky/slp3/9.pdf)
- [Illustrating Reinforcement Learning from Human Feedback (RLHF) ](https://huggingface.co/blog/rlhf)
- [Simplifying Alignment: From RLHF to Direct Preference Optimization (DPO)](https://huggingface.co/blog/ariG23498/rlhf-to-dpo)
- [DPO paper: Direct Preference Optimization](https://arxiv.org/abs/2305.18290)
- [Hugging Face TRL `DPOTrainer` docs](https://huggingface.co/docs/trl/en/dpo_trainer)
- [Hugging Face TRL dataset formats](https://huggingface.co/docs/trl/main/dataset_formats)
- [Hugging Face PEFT LoRA guide](https://huggingface.co/docs/peft/en/developer_guides/lora)

In [ ]:
import sys
!{sys.executable} -m pip install -U transformers datasets accelerate trl peft torchao requests


## Alignment

Today, we will look at **alignment**: how we can make a language model's behavior better match human preferences.

A language model can be fluent and still be **misaligned**.

Examples:
- it may confidently invent facts instead of saying "I don't know"
- it may ignore instructions about tone or safety
- it may optimize for sounding helpful rather than being correct

Alignment is about steering the model toward the kinds of responses humans actually want.


### Group Exercise

Check out this dataset which can be used for alignment training: https://huggingface.co/datasets/Anthropic/hh-rlhf

**Content Warning:** This dataset contains extremely offensive and upsetting content so that models can explicitly learn what **not** to do.

**Discuss:** What are some *alignment goals* that appear to be behind this data? What are the dataset creators trying to do with model that are trained on this data?



### Discuss

What are some examples of bad things that could happen from a misaligned system?

Come up with at least one **big** problem and one **small** problem.


## Reinforcement Learning

**Reinforcement Learning** is a machine learning paradigm in which an agent improves its behavior based on rewards and penalties.

In language-model post-training, the "reward" is usually not built into the environment. Instead, researchers often try to estimate what humans would prefer.

<div>
    <center>
    <img src="https://github.com/ericmanley/s26-CS195NLP/blob/main/images/Reinforcement_learning_diagram.svg.png?raw=1" width="500">
    </center>
</div>

image source: https://en.wikipedia.org/wiki/File:Reinforcement_learning_diagram.svg


## Reinforcement Learning from Human Feedback (RLHF)

A traditional RLHF pipeline has two major stages:

1. Collect **human preferences** between pairs of model outputs.
2. Use those preferences to train the model toward preferred behavior.

A common version introduces a separate **reward model** that tries to predict which outputs humans would prefer.

<div>
    <center>
        <img src="https://github.com/ericmanley/s26-CS195NLP/blob/main/images/reward-model.png?raw=1" width="700">
    </center>
</div>

image source: https://huggingface.co/blog/rlhf


## Fine Tuning the language model from the reward model

<div>
    <center>
        <img src="https://github.com/ericmanley/s26-CS195NLP/blob/main/images/rlhf.png?raw=1" width=700>
    </center>
</div>

### how it works

1. Make a copy of the initial language model, and freeze some parameters
    - freezing parameters helps make sure you don't mess up some of the fundamental language understanding represented by earlier layers
    - faster, less expensive
    - will be updated with the Proximal Policy Optimization - a RL algorithm
2. Use the *reward model* to predict the output's preferability rating
3. Compute a penalty $\lambda_{r_{KL}}$ based on how far away the fine-tuned output diverges from the initial model
    - without this, the model can end up generating gibberish that fools the reward model
    - calculated using Kullback-Leibler divergence - basically calculate one probability distribution's distance/loss relative to the other
4. Update the weights in the new model, aiming toward better reward

image source: https://huggingface.co/blog/rlhf

## We're not going to do full RLHF today

Full RLHF is historically important, but there's a lot to get right

A full pipeline often includes:
- a policy model
- a reward model
- multiple sampled outputs per prompt
- an RL optimization algorithm such as PPO
- extra tricks to keep the model from drifting too far

## DPO: A Simpler Preference-Tuning Method

**Direct Preference Optimization (DPO)** is a simpler alternative.

Instead of first training a separate reward model and then running RL, DPO trains directly on **preference pairs** using a **Maximum Likelihood Estimation** method in which the model parameters are adjusted to make the preferred responses more probable.
* see the math here: https://huggingface.co/blog/ariG23498/rlhf-to-dpo

Each training example has:
- a `prompt`
- a `chosen` response
- a `rejected` response

*View the training data here:* https://github.com/ericmanley/S26-CS195NLP/blob/main/data/course_advising_dpo_preferences.json


## Alignment Goal for This Workshop

Let's continue with our RAG course dataset that we worked on a couple weeks ago with this alignment goal in mind:

> Make a course assistant that prefers **grounded, honest, context-based answers** over hallucinated or overconfident ones.


A better aligned answer here should:
- answer from the provided course context
- avoid adding unsupported details
- say when the context does not contain enough information


In [ ]:
import copy
import json
import random

import requests
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig

random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Using device:', device)


Using device: cuda


## Choose a Base Model

We'll try using `Qwen/Qwen2.5-0.5B-Instruct` again


In [ ]:
checkpoint = 'Qwen/Qwen2-7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint)
model.to(device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [ ]:
SYSTEM_MESSAGE = (
    'You are a careful academic advising assistant. '
    'Answer using only the course information in the provided context. '
    'If the context does not contain the answer, say so clearly. '
    'Do not invent course facts, instructors, times, or prerequisites.'
)


def make_chat_messages(prompt):
    return [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user', 'content': prompt},
    ]


def generate_chat_response(messages, model=model, tokenizer=tokenizer, max_new_tokens=120):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:],skip_special_tokens=True)



In [ ]:
def print_context_summary(prompt_text, max_description_chars=180):
    import re
    course_blocks = re.split(r'\nRetrieved course \d+[^\n]*\n', prompt_text)[1:]
    if not course_blocks:
        print('Context: [no retrieved course information found]')
        return

    print('CONTEXT:')
    for block in course_blocks:
        lines = block.splitlines()
        course = title = faculty = times = location = description = ''
        for line in lines:
            if line.startswith('Course:'):
                course = line.removeprefix('Course:').strip()
            elif line.startswith('Title:'):
                title = line.removeprefix('Title:').strip()
            elif line.startswith('Faculty:'):
                faculty = line.removeprefix('Faculty:').strip() or 'not listed'
            elif line.startswith('Times:'):
                times = line.removeprefix('Times:').strip() or 'not listed'
            elif line.startswith('Location:'):
                location = line.removeprefix('Location:').strip() or 'not listed'
            elif line.startswith('Description:'):
                description = line.removeprefix('Description:').strip()

        description = ' '.join(description.split())
        if len(description) > max_description_chars:
            description = description[:max_description_chars].rstrip() + '...'

        heading = f'- {course}: {title}'.strip()
        print(heading)
        print(f'  Faculty: {faculty}; Times: {times}; Location: {location}')
        if description:
            print(f'  Description: {description}')


## Load the Course Advising Preference Dataset

How the dataset was made:

1. I had GPT5.5 look at the course data file (`data/f25_course_information.json`) and generate some questions that can be used with the RAG workflow we built a couple weeks ago (`F5_1`)
2. Using the RAG workflow, we generated two responses for each question, using different temperatures to get variation in the responses
3. I had GPT5.5 filter out some examples where both answers were bad, where there was not enough variation between the two answers, and then I asked it to write a few "fake" model responses so we can maybe push the model to learn that behavior (like, say when the answer isn't available in the context)
4. I had GPT5.5 pick the better response as `chosen` and the worse response as `rejected` based on our alignment goals



In [ ]:
preference_url = 'https://raw.githubusercontent.com/ericmanley/S26-CS195NLP/refs/heads/main/data/course_advising_dpo_preferences.json'
response = requests.get(preference_url)
preference_rows = response.json()

# Local fallback if you are working offline:
# with open('data/course_advising_dpo_preferences.json', 'r', encoding='utf-8') as f:
#     preference_rows = json.load(f)


formatted_rows = [
    {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': row['prompt']},
        ],
        'chosen': [
            {'role': 'assistant', 'content': row['chosen']},
        ],
        'rejected': [
            {'role': 'assistant', 'content': row['rejected']},
        ],
    }
    for row in preference_rows
]

random.shuffle(formatted_rows)

# Keep a held-out slice for qualitative checks after training.
eval_size = 15
train_rows = formatted_rows[:-eval_size]
eval_rows = formatted_rows[-eval_size:]

train_dataset = Dataset.from_list(train_rows)
eval_dataset = Dataset.from_list(eval_rows)

print('train examples:', len(train_dataset))
print('eval examples:', len(eval_dataset))
display('Example from training set:',train_dataset[0])


train examples: 60
eval examples: 15


'Example from training set:'

{'prompt': [{'role': 'system',
   'content': 'You are a careful academic advising assistant. Answer using only the course information in the provided context. If the context does not contain the answer, say so clearly. Do not invent course facts, instructors, times, or prerequisites.'},
  {'role': 'user',
   'content': 'You are a careful course advising assistant. Use only the retrieved Drake course information. If the context does not explicitly answer the question, say that you cannot determine the answer from the provided course information.\n\nQuestion: Can MATH 080 satisfy an elective requirement for my computer science major?\n\nRetrieved course information:\nRetrieved course 1 (similarity score: 0.514)\nCourse: MATH 054\nSubject: Mathematics\nTitle: DISCRETE MATHEMATICS\nDescription: Number systems, algorithms, set theory, logic, Boolean algebra, functions, combinatorics, probability, graph theory.  Prereq.:  MATH 20 or equivalent.\nPrerequisites: Prerequisite(s): MATH 020 or MA

## Examples from the Real Dataset

Here are a few examples that show the behavior we are trying to align toward.

```json
  {
    "id": "q_0065",
    "category": "messy_student_wording",
    "prompt": "You are a careful course advising assistant. Use only the retrieved Drake course information.\n\nQuestion: what's the class for assembly stuff? professor too please\n\nRetrieved course information:\nRetrieved course 1 (similarity score: 0.567)\nCourse: CS 130\nSubject: Computer Sciences\nTitle: COMPUTER ORGANIZATION AND ASSEMBLY\nDescription: Computer organization and architecture; internal representation of programs and data;  assembly language programming; addressing techniques, macros, assemblers, linking; input/output concepts.  Prereq.:  CS 66 or equivalent.\nPrerequisites: Prerequisite(s): CS 066\nFaculty: Eric Manley\nAttributes: \nLocation: C-S 0301\nTimes: Monday, Wednesday 1230-1345\n\nRetrieved course 2 (similarity score: 0.564)\nCourse: CS 130\nSubject: Computer Sciences\nTitle: COMPUTER ORGANIZATION AND ASSEMBLY\nDescription: Computer organization and architecture; internal representation of programs and data;  assembly language programming; addressing techniques, macros, assemblers, linking; input/output concepts.  Prereq.:  CS 66 or equivalent.\nPrerequisites: Prerequisite(s): CS 066\nFaculty: Eric Manley\nAttributes: \nLocation: C-S 0301\nTimes: Monday, Wednesday 1400-1515\n\nRetrieved course 3 (similarity score: 0.479)\nCourse: CS 067\nSubject: Computer Sciences\nTitle: OBJECT-ORIENTED PROGRAMMING\nDescription: This course introduces students to object-oriented programming (OOP). Students will learn OOP concepts such as classes, objects, encapsulation, messaging, data hiding, inheritance, and polymorphism. Generic programming and OOP design patterns will also be taught. Students will encounter advanced programming projects where unit testing and exception handling will be stressed. Other topics include serialization and GUI construction. Prereq.: CS 066.\nPrerequisites: Prerequisite(s): CS 066\nFaculty: \nAttributes: \nLocation: C-S 0301\nTimes: Tuesday, Thursday 1100-1215\n\nAnswer the question using only the retrieved course information.",
    "chosen": "CS 130 - Computer Organization and Assembly - Professor: Eric Manley",
    "rejected": "CS 067 - OBJECT-ORIENTED PROGRAMMING",
    "chosen_label": "answer_a",
    "preference_note": "A directly identifies CS 130 and Eric Manley; B gives the wrong course.",
    "source_dataset": "general_rag_candidates_first_pass",
    "preference_id": "p_0036"
  },
    {
    "id": "u_0006",
    "category": "unsupported_registration",
    "prompt": "You are a careful course advising assistant. Use only the retrieved Drake course information. If the context does not explicitly answer the question, say that you cannot determine the answer from the provided course information.\n\nQuestion: Is there a waitlist for CS 167 Machine Learning?\n\nRetrieved course information:\nRetrieved course 1 (similarity score: 0.542)\nCourse: CS 167\nSubject: Computer Sciences\nTitle: MACHINE LEARNING\nDescription: This course introduces approaches to developing computer programs that learn from data.  Both foundational and contemporary machine learning algorithms will be covered in the context of a variety of data and problem types.  Specific topics will vary but may include artificial neural networks, decision trees, instance-based learning, Bayesian learning, support vector machines, hidden Markov models, reinforcement learning, and natural language processing.  Students will develop their own implementations of the algorithms as well as utilize modern machine learning software and programming libraries.  Pre-requisite: CS 065 and (CS 066 or STAT 040).\nPrerequisites: Prerequisite(s): CS 065 and (CS 066 or STAT 040)\nFaculty: \nAttributes: \nLocation: C-S 0135\nTimes: Tuesday, Thursday 1400-1515\n\nRetrieved course 2 (similarity score: 0.540)\nCourse: CS 167\nSubject: Computer Sciences\nTitle: MACHINE LEARNING\nDescription: This course introduces approaches to developing computer programs that learn from data.  Both foundational and contemporary machine learning algorithms will be covered in the context of a variety of data and problem types.  Specific topics will vary but may include artificial neural networks, decision trees, instance-based learning, Bayesian learning, support vector machines, hidden Markov models, reinforcement learning, and natural language processing.  Students will develop their own implementations of the algorithms as well as utilize modern machine learning software and programming libraries.  Pre-requisite: CS 065 and (CS 066 or STAT 040).\nPrerequisites: Prerequisite(s): CS 065 and (CS 066 or STAT 040)\nFaculty: \nAttributes: \nLocation: SCB 0301\nTimes: Tuesday, Thursday 1530-1645\n\nRetrieved course 3 (similarity score: 0.493)\nCourse: CS 066\nSubject: Computer Sciences\nTitle: INTRODUCTION TO COMPUTER SCIENCE II\nDescription: Continuance of CS 65 using a block-structured language and emphasizing data abstraction. More general data structures and alternative implementations of them are used in programs, Sorting, searching and tree traversal algorithms are used and analyzed. Provides preparation for further study in computer science. Prereq: CS 65 or equivalent\nPrerequisites: Prerequisite(s): CS 065\nFaculty: Adam Case\nAttributes: \nLocation: C-S 0301\nTimes: Tuesday, Thursday 1400-1515\n\nAnswer the question using only the retrieved course information.",
    "chosen": "The retrieved course information describes CS 167, but it does not mention whether there is a waitlist. I cannot determine waitlist status from the provided course information; check the registration system or department.",
    "rejected": "Course number: CS 167\nYes, there is a waitlist for CS 167 Machine Learning, which is described as \"This course introduces approaches to developing computer programs that learn from data. Both foundational and contemporary machine learning algorithms will be covered in the context of a variety of data and problem types. Specific topics will vary but may include artificial neural networks, decision trees, instance-based learning, Bayesian learning, support vector machines, hidden Markov models, reinforcement learning, and natural language processing. Students will develop their own implementations of the algorithms as well as utilize modern machine learning software and programming libraries.\" The prerequisite for this course is CS 065 and (CS 066 or STAT 040).",
    "rejected_label": "answer_b",
    "preference_note": "Prefer acknowledging that the retrieved context does not contain policy/advising/current-status information over making an unsupported claim.",
    "source_dataset": "targeted_unsupported_context",
    "preference_id": "p_0049"
  },
  ```


## Before Training: Try a Few Held-Out Prompts

These prompts come from the held-out split of the real preference dataset. The model will see the retrieved context and question, but we will not show it the preferred answer.


In [ ]:
heldout_examples = list(eval_dataset.select(range(min(4, len(eval_dataset)))))

for row in heldout_examples:
    print('=' * 80)
    user_message = next(
        message['content'] for message in row['prompt']
        if message['role'] == 'user'
    )
    question_line = next(
        (line for line in user_message.splitlines() if line.startswith('Question:')),
        'Question: [not found]'
    )
    print(question_line)
    print_context_summary(user_message)
    print('\nMODEL ANSWER:')
    print(generate_chat_response(row['prompt']))
    print('\nPREFERRED ANSWER FROM DATASET:')
    print(row['chosen'][0]['content'])
    print()


Question: Are any math or statistics courses listed with the Quantitative Literacy attribute?
CONTEXT:
- JBC 034: TOPICS IN DATA AND STAT LIT
  Faculty: not listed; Times: not listed; Location: not listed
  Description: This seminar component develops students’ ability to understand, interpret, and critically examine information expressed as statistics and both quantitative and qualitative data; t...
- STAT 072: STATISTICS II
  Faculty: Amy Vaughan; Times: Tuesday, Thursday 1100-1215; Location: ALIB 0004
  Description: Continuance of STAT 071 with further tests of significance; analysis of variance; correlation and regression; and contingency table analysis. Prereq.: STAT 071, STAT 130, or ACTS 1...
- STAT 072: STATISTICS II
  Faculty: Amy Vaughan; Times: Tuesday, Thursday 1230-1345; Location: ALIB 0004
  Description: Continuance of STAT 071 with further tests of significance; analysis of variance; correlation and regression; and contingency table analysis. Prereq.: STAT 071, STAT 130,

## What Is TRL and `DPOTrainer`?

In the fine-tuning notebook (`F6_4`), we used `TRL`'s `SFTTrainer` for supervised fine-tuning.

Here we will use `TRL`'s `DPOTrainer` instead.

`DPOTrainer` handles several pieces for us:
- tokenizing preference examples
- comparing chosen vs rejected responses for the same prompt
- computing the DPO loss
- running the training loop
- optionally attaching LoRA adapters through PEFT

So the overall workflow should feel familiar even though the training signal is different.


## LoRA Setup

As before, we will fine-tune with **LoRA adapters** rather than updating every model parameter.



In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

training_args = DPOConfig(
    output_dir='f7_2_alignment_dpo_lora',
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    logging_steps=1,
    save_strategy='no',
    eval_strategy='no',
    report_to='none',
    max_length=512,
    beta=0.1,
    fp16=torch.cuda.is_available(),
)

training_args


DPOConfig(output_dir='f7_2_alignment_dpo_lora', per_device_train_batch_size=1, num_train_epochs=2, max_steps=-1, learning_rate=1e-05, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs=None, warmup_steps=0, optim=<OptimizerNames.ADAMW_TORCH_FUSED: 'adamw_torch_fused'>, optim_args=None, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=4, average_tokens_across_devices=True, max_grad_norm=1.0, label_smoothing_factor=0.0, bf16=False, fp16=True, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=None, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=1, logging_first_step=False, log_on_each_node

## Fine-Tune with DPO

This may take a few minutes on a Colab GPU.


In [ ]:
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

model.train() #switch to train mode
trainer.train()


Tokenizing train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.693147
2,0.693147
3,0.693147
4,0.693799
5,0.693891
6,0.693147
7,0.693147
8,0.693147
9,0.693147
10,0.692426


TrainOutput(global_step=30, training_loss=0.6906388541062672, metrics={'train_runtime': 41.7945, 'train_samples_per_second': 2.871, 'train_steps_per_second': 0.718, 'total_flos': 5204350026215424.0, 'train_loss': 0.6906388541062672})

## After DPO: Ask the Held-Out Questions Again

We are looking for whether the model becomes more likely to:
- stay within the provided context
- avoid unsupported guesses
- explicitly say when information is missing

This is a qualitative check, similar in spirit to the one we did after SFT in `F6_4`.


In [ ]:
model.eval()
for row in heldout_examples:
    print('=' * 80)
    user_message = next(
        message['content'] for message in row['prompt']
        if message['role'] == 'user'
    )
    question_line = next(
        (line for line in user_message.splitlines() if line.startswith('Question:')),
        'Question: [not found]'
    )
    print(question_line)
    print_context_summary(user_message)
    print('\nMODEL ANSWER:')
    print(generate_chat_response(row['prompt'], model=model, tokenizer=tokenizer))
    print('\nPREFERRED ANSWER FROM DATASET:')
    print(row['chosen'][0]['content'])
    print()


Question: Are any math or statistics courses listed with the Quantitative Literacy attribute?
CONTEXT:
- JBC 034: TOPICS IN DATA AND STAT LIT
  Faculty: not listed; Times: not listed; Location: not listed
  Description: This seminar component develops students’ ability to understand, interpret, and critically examine information expressed as statistics and both quantitative and qualitative data; t...
- STAT 072: STATISTICS II
  Faculty: Amy Vaughan; Times: Tuesday, Thursday 1100-1215; Location: ALIB 0004
  Description: Continuance of STAT 071 with further tests of significance; analysis of variance; correlation and regression; and contingency table analysis. Prereq.: STAT 071, STAT 130, or ACTS 1...
- STAT 072: STATISTICS II
  Faculty: Amy Vaughan; Times: Tuesday, Thursday 1230-1345; Location: ALIB 0004
  Description: Continuance of STAT 071 with further tests of significance; analysis of variance; correlation and regression; and contingency table analysis. Prereq.: STAT 071, STAT 130,

## Discussion

How did this do? Are there any noticable changes? If so, what?


its still too confident in its answer but its isnt rambling with fake information as much any more.

## Discussion

In what situations is it useful to do Supervised Fine Tuning (SFT)?

In what situations is it useful to do Direct Preference Optimization?



## Applied Exploration

Get this code working with another preference data set like https://huggingface.co/datasets/Anthropic/hh-rlhf

You can experiment with a different model if you like, and you may also use a subset of any dataset given computing limits.

Describe:
* Any changes you made
* What did you have to change to make it work?
* What kind of results did you notice?


## Creative Synthesis Idea

* Experiment more with the dataset in this notebook, and work towards enhancing the project to make meaningful alignment differences

* Add an alignment step to some other model you've built into an application or fine-tuned this semester

* Build an application that builds preference data sets by allowing users to vote on different model outputs

